In [0]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Generate unique identifiers for this ETL run
SSIDGUID = str(uuid.uuid4())
ETLRunTime = datetime.now()
IncrementalExtractTime = datetime.now()

# Configuration variables matching the original stored procedure
MasterSourceSystemName = ('Excel')
SourceSubSystemName = ('')
SourceDatabaseName = ('DJJStarSchema')
SourceTableName = ('Nue.RecyclingYardInventoryPrepared')
DestinationSchemaName = ('NUE')
DestinationTableName = ('factRecyclingYardInventoryRatio')
DestinationDatabaseName = ('DJJStarSchema')
ETLType = ('Incremental')
BatchSize = 1000
PkgName = ('NUE.etl_factRecyclingYardInventoryRatio')
ETLTemplate = ('')
PkgGUID = str(uuid.uuid4())
DJJLastUpdateTime = datetime.now()

# Catalog mappings
# source_catalog: bronze layer for DJJODS operational source (DJJODS -> bronze_non_prod)
source_catalog = "bronze_non_prod"
# target_catalog: enriched/temporary layer for temp tables and target objects
target_catalog = "djj_enriched_non_prod"
# federated_starschema_catalog: star schema layer (DJJStarSchema -> djj_starschema)
federated_starschema_catalog = "djj_starschema"

# Audit counters
Success = '0'
RowsInserted = 0
RowsUpdated = 0
RowsDeleted = 0
ErrorRowCount = 0
TableInitialRowCount = 0
TableFinalRowCount = 0
ETLStatus = ('')
ExtractRowCount = 0
SSISGUID = SSIDGUID

### Execute the Initial Metadata Logging

In [0]:
%run /Workspace/Shared/metadata_framework/metadata_logger

In [0]:
# Insert/Update records in ETL Master
logger = ETLLogger(spark)

ETLID = logger.log_etl(
    MasterSourceSystemName=MasterSourceSystemName,
    SourceSubSystemName=SourceSubSystemName,
    SourceDatabaseName=SourceDatabaseName,
    SourceTableName=SourceTableName,
    DestinationDatabaseName=DestinationDatabaseName,
    DestinationTableName=DestinationTableName,
    PkgName=PkgName,
    PkgGUID=PkgGUID,
    ETLTemplate=ETLTemplate,
    ETLType=ETLType,
    CheckpointsEnabled=True
)

print(ETLID)

In [0]:
# Insert records in ETLExecutionLog table
exec_logger = ETLExecutionLogger(spark)
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success=Success,
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [0]:
ETLExecutionID = spark.sql(f"""SELECT ETLExecutionID from {target_catalog}.metadata.etlexecutionlog where lower(trim(ETLID)) = lower(trim('{ETLID}')) and lower(trim(SSISGUID)) = lower(trim('{SSIDGUID}'))""").first()[0]

### ETL Logic Starts Here

In [0]:
# Get initial row count of target table
TableInitialRowCount = spark.sql(f"SELECT COUNT(1) AS cnt FROM {target_catalog}.NUE.factRecyclingYardInventoryRatio").first()["cnt"]
print(f"Initial row count: {TableInitialRowCount}")

In [0]:
# Get TargetLastUpdatedDate
# Original: SELECT @TargetLastUpdatedDate = ISNULL(MAX(CAST(DJJLastUpdateTime AS DATE)), '1900-01-01')
# FROM DJJStarSchema.Nue.factRecyclingYardInventoryRatio
# DJJLastUpdateTime -> EnrichedTimestampUTC
target_last_updated_row = spark.sql(f"""
    SELECT COALESCE(CAST(MAX(EnrichedTimestampUTC) AS DATE), CAST('1900-01-01' AS DATE)) AS TargetLastUpdatedDate
    FROM {target_catalog}.NUE.factRecyclingYardInventoryRatio
""").first()
TargetLastUpdatedDate = str(target_last_updated_row["TargetLastUpdatedDate"])
print(f"TargetLastUpdatedDate: {TargetLastUpdatedDate}")

In [0]:
# Step 1: Build intermediate date-range table (#InventoryYardRatiosDates)
# Uses ROW_NUMBER() and LAG() to compute DateEnded for each (InventoryCode, Yard) partition
# Source: DJJODS.ref.ODS_NUE_RecyclingYardInventoryPrepared -> {source_catalog}.ref.ODS_NUE_RecyclingYardInventoryPrepared
# DATEADD(dd, -1, LAG(...)) -> DATE_SUB(LAG(...), 1)
# ISNULL() -> COALESCE()

# First create with ROW_NUMBER
spark.sql(f"""
    CREATE TABLE {target_catalog}.temp.InventoryYardRatiosDates AS
    SELECT
        x.InventoryCode,
        x.Yard,
        x.DateEntered,
        CAST(NULL AS DATE) AS DateEnded,
        ROW_NUMBER() OVER (PARTITION BY x.InventoryCode, x.Yard ORDER BY x.DateEntered DESC) AS RowNumber
    FROM (
        SELECT DISTINCT
            InventoryCode,
            Yard,
            DateEntered
        FROM {source_catalog}.ref.ODS_NUE_RecyclingYardInventoryPrepared
    ) x
""")
print("Intermediate table InventoryYardRatiosDates created.")

In [0]:
# Step 2: Compute DateEnded using LAG window function
# Original: UPDATE a SET a.DateEnded = ISNULL(b.DateEnded, '3000-01-01')
# WHERE b.DateEnded = DATEADD(dd, -1, LAG(DateEntered) OVER (PARTITION BY InventoryCode, Yard ORDER BY RowNumber))
spark.sql(f"""
    MERGE INTO {target_catalog}.temp.InventoryYardRatiosDates a
    USING (
        SELECT
            InventoryCode,
            Yard,
            DateEntered,
            DATE_SUB(LAG(DateEntered) OVER (PARTITION BY InventoryCode, Yard ORDER BY RowNumber), 1) AS DateEnded
        FROM {target_catalog}.temp.InventoryYardRatiosDates
    ) b
    ON a.InventoryCode = b.InventoryCode
        AND lower(trim(a.Yard)) = lower(trim(b.Yard))
        AND a.DateEntered = b.DateEntered
    WHEN MATCHED THEN UPDATE SET
        a.DateEnded = COALESCE(b.DateEnded, CAST('3000-01-01' AS DATE))
""")
print("DateEnded computed for InventoryYardRatiosDates.")

In [0]:
# Step 3: Build the fact temp table
# Source: DJJODS.ref.ODS_NUE_RecyclingYardInventoryPrepared -> {source_catalog}.ref.ODS_NUE_RecyclingYardInventoryPrepared
# Lookup: DJJ.dimLocation -> {federated_starschema_catalog}.DJJ.dimLocation
#   Special handling: RMR-Harrison/Spring Grove only match when LocationID IN ('HR','SG')
#   Other locations match normally
# GETDATE() -> current_date()
# DJJLastUpdateTime -> EnrichedTimestampUTC

spark.sql(f"""
    CREATE TABLE {target_catalog}.temp.factRecyclingYardInventoryRatio AS
    SELECT
        a.InventoryCode,
        CASE
            WHEN lower(trim(a.Yard)) = lower(trim('Default')) THEN 'DEFAULT'
            ELSE c.LocationID
        END AS LocationID,
        a.Yard,
        a.Status,
        CAST(CAST(a.PercentPrepared AS STRING) AS DOUBLE) AS PercentPrepared,
        a.DateEntered,
        b.DateEnded,
        CASE
            WHEN current_date() BETWEEN a.DateEntered AND b.DateEnded THEN 1
            ELSE 0
        END AS ActiveTimePeriodFlag,
        0 AS DJJGeneratedFlag,
        {ETLExecutionID} AS ETLSSExecutionID,
        current_timestamp() AS EnrichedTimestampUTC
    FROM {source_catalog}.ref.ODS_NUE_RecyclingYardInventoryPrepared a
        INNER JOIN {target_catalog}.temp.InventoryYardRatiosDates b
            ON a.InventoryCode = b.InventoryCode
            AND lower(trim(a.Yard)) = lower(trim(b.Yard))
            AND a.DateEntered = b.DateEntered
        LEFT OUTER JOIN (
            SELECT LocationKey, LocationID, Location
            FROM {federated_starschema_catalog}.DJJ.dimLocation
            WHERE Location IN ('RMR-Harrison', 'RMR-Spring Grove')
                AND LocationID IN ('HR', 'SG')
            UNION
            SELECT LocationKey, LocationID, Location
            FROM {federated_starschema_catalog}.DJJ.dimLocation
            WHERE Location NOT IN ('RMR-Harrison', 'RMR-Spring Grove')
        ) c
            ON lower(trim(a.Yard)) = lower(trim(c.Location))
""")
print("Temp table factRecyclingYardInventoryRatio created and populated.")

### UPSERT Logic

In [0]:
# Perform Updates - MERGE INTO target using temp table
# Original SQL uses UPDATE ... FROM with INNER JOIN on (DateEntered, InventoryCode, Yard)
# DJJLastUpdateTime -> EnrichedTimestampUTC
spark.sql(f"""
    MERGE INTO {target_catalog}.NUE.factRecyclingYardInventoryRatio a
    USING {target_catalog}.temp.factRecyclingYardInventoryRatio b
    ON a.DateEntered = b.DateEntered
        AND a.InventoryCode = b.InventoryCode
        AND lower(trim(a.Yard)) = lower(trim(b.Yard))
    WHEN MATCHED THEN UPDATE SET
        a.Yard = b.Yard,
        a.Status = b.Status,
        a.PercentPrepared = b.PercentPrepared,
        a.DateEnded = b.DateEnded,
        a.ActiveTimePeriodFlag = b.ActiveTimePeriodFlag,
        a.DJJGeneratedFlag = b.DJJGeneratedFlag,
        a.ETLSSExecutionID = b.ETLSSExecutionID,
        a.EnrichedTimestampUTC = b.EnrichedTimestampUTC
""")
print("Updates completed.")

In [0]:
# Get RowsUpdated count from the MERGE operation
history_df = spark.sql(f"DESCRIBE HISTORY {target_catalog}.NUE.factRecyclingYardInventoryRatio LIMIT 1")
operation_metrics = history_df.select("operationMetrics").first()[0]
RowsUpdated = int(operation_metrics.get("numTargetRowsUpdated", 0)) if operation_metrics else 0
print(f"Rows updated: {RowsUpdated}")

In [0]:
# Perform Inserts - Insert new records not already in the target
# Original SQL uses LEFT OUTER JOIN on (DateEntered, InventoryCode, Yard) WHERE b.DateEntered IS NULL
# DJJLastUpdateTime -> EnrichedTimestampUTC
spark.sql(f"""
    INSERT INTO {target_catalog}.NUE.factRecyclingYardInventoryRatio (
        InventoryCode,
        LocationID,
        Yard,
        Status,
        PercentPrepared,
        DateEntered,
        DateEnded,
        ActiveTimePeriodFlag,
        DJJGeneratedFlag,
        ETLSSExecutionID,
        EnrichedTimestampUTC
    )
    SELECT
        a.InventoryCode,
        a.LocationID,
        a.Yard,
        a.Status,
        a.PercentPrepared,
        a.DateEntered,
        a.DateEnded,
        a.ActiveTimePeriodFlag,
        a.DJJGeneratedFlag,
        a.ETLSSExecutionID,
        a.EnrichedTimestampUTC
    FROM {target_catalog}.temp.factRecyclingYardInventoryRatio a
        LEFT OUTER JOIN {target_catalog}.NUE.factRecyclingYardInventoryRatio b
            ON a.DateEntered = b.DateEntered
            AND a.InventoryCode = b.InventoryCode
            AND lower(trim(a.Yard)) = lower(trim(b.Yard))
    WHERE b.DateEntered IS NULL
""")
print("Inserts completed.")

In [0]:
# Get RowsInserted count from the INSERT operation
history_df = spark.sql(f"DESCRIBE HISTORY {target_catalog}.NUE.factRecyclingYardInventoryRatio LIMIT 1")
operation_metrics = history_df.select("operationMetrics").first()[0]
RowsInserted = int(operation_metrics.get("numOutputRows", 0)) if operation_metrics else 0
print(f"Rows inserted: {RowsInserted}")

In [0]:
# No deletes in this ETL
RowsDeleted = 0
print(f"Rows deleted: {RowsDeleted}")

### Close Out the ETL - Final Audit & Logging

In [0]:
# Get final rowcount of target table
final_count_row = spark.sql(f"""SELECT COUNT(1) AS cnt FROM {target_catalog}.NUE.factRecyclingYardInventoryRatio""").first()
TableFinalRowCount = final_count_row['cnt'] if final_count_row else 0
print(f"Final row count: {TableFinalRowCount}")
print(f"Rows added: {TableFinalRowCount - TableInitialRowCount}")

In [0]:
# Update records in ETLExecutionLog table with success status
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success='1',
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [0]:
# Update ETLMaster with final metadata
spark.sql(f"""UPDATE {target_catalog}.metadata.ETLMaster
    SET lower(trim(IncrementalExtractTime)) = lower(trim('{IncrementalExtractTime}')), 
        lower(trim(ETLTemplate)) = lower(trim('{ETLTemplate}')), 
        lower(trim(ETLType)) = lower(trim('{ETLType}'))
    WHERE lower(trim(ETLID)) = lower(trim('{ETLID}'))""")

In [0]:
# Cleanup - drop temp tables
spark.sql(f"DROP TABLE IF EXISTS {target_catalog}.temp.InventoryYardRatiosDates")
spark.sql(f"DROP TABLE IF EXISTS {target_catalog}.temp.factRecyclingYardInventoryRatio")
print("Temp tables cleaned up. ETL completed successfully.")